# plan-a-closure Tier 0 on a Colab GPU (engineering shakedown)

This runs the one-command Tier 0 pilot (`closure tier0`) on the GPU this Colab runtime has, so the pipeline is checked on
a **real device** instead of the fake NVML used in CI.

**What it is.** A shakedown on a non-target device, inside a virtual machine, with NVML as the only power source.
Precisions the GPU lacks (fp8 before compute capability 8.9) are dropped and noted in the logs.

**What it is not.** The registered trial. There is no independent power meter, so assumption A6 does not hold, and the
registered numbers come from Tier 1 on instrumented H100/H200 hardware (`docs/PREREGISTRATION.md`). Every run is
published whatever it shows (`docs/DEVIATIONS.md` has the earlier ones).

**How to run it**

1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Runtime > **Run all**.
3. Approve the Google Drive prompt in cell 1, so the results are also saved to `MyDrive/plan-a-closure-runs/`
   (decline it and they still download to this computer when the run ends).
4. Keep this tab open and the computer awake for about 5 hours. When cell 3 finishes, the results are already
   saved: nothing is lost if Colab disconnects afterwards. If Colab disconnects before that, the run is lost: run it
   again, or set `PROFILE = "smoke"` in cell 3 for the short version.

Free Colab limits GPU time per day. If it says no GPU is available, try again later.

**Which trial.** `PROFILE` in cell 3 chooses it. Both have the same design (calibration blocks interleaved with a
balanced blind audit, the link analysis, the hidden load measured on its own first; amendment v0.6).

| `PROFILE` | whole run | trial | what it can decide |
| --- | --- | --- | --- |
| `"medium"` (default) | about 5 h | 3.8 h: 150 reference and 300 audit blocks of 30 s | the registered rules H1.1 to H1.4 at shakedown scale: 204 null blocks, and the false-alarm rule needs 200 |
| `"smoke"` | about 2 h 20 min | 1 h: 100 reference and 80 audit blocks of 20 s | that the pipeline works; 40 null blocks cannot decide the false-alarm rule |

Colab lets the run lock the GPU's clocks, which Kaggle's container does not, so this is where the time bound is checked
with clocks held still and where the probe can count clock settings among the stack variants. For two long trials
read together (about 10.5 hours, blocks of 60 s), use `notebooks/tier0_kaggle.ipynb` on Kaggle: a free Colab session
is not reliable for that long.

In [ ]:
# 1. Setup: the GPU, Google Drive (optional), the exact code, persistence mode.
import os, shutil, subprocess, time
REPO, REF = "https://github.com/AnonRish/plan-a-closure", "main"   # REF can be a tag or a commit, to pin the code
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU attached: Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,uuid,driver_version,memory.total,power.limit", "--format=csv"],
                     capture_output=True, text=True).stdout)
SAVE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/plan-a-closure-runs"
    os.makedirs(SAVE, exist_ok=True)
    print("results will also be saved to Google Drive:", SAVE)
except Exception as e:
    print(f"Google Drive not mounted ({type(e).__name__}): the results will download to this computer at the end")
if not os.path.isdir("/content/plan-a-closure/.git"):
    !git clone -q --depth 1 {REPO} /content/plan-a-closure
!git -C /content/plan-a-closure fetch -q --depth 1 origin {REF} && git -C /content/plan-a-closure checkout -q FETCH_HEAD
%cd /content/plan-a-closure
!pip install -q -e ".[test,hw]"
!pip install -q opentimestamps-client    # stamps the pre-registration on public calendars before any trial data exist
!python -c "import closure; print('plan-a-closure', closure.__version__)"
!git log -1 --format="code: commit %H (%cd)"
!nvidia-smi -pm 1 > /dev/null && echo "persistence mode on"

In [ ]:
# 2. Preflight: NVML, the energy counter, torch and CUDA, which physical GPU. FAIL lines here stop the run in cell 3;
#    WARN lines for fp8, clock sync and tcpdump are normal on Colab.
!closure preflight --gpu 0

## The run (about 5 hours; about 2 hours 20 minutes with `PROFILE = "smoke"`)

Cell 3 runs Tier 0, then A Narrow Path's mock training run on the same GPU, then saves everything to Google Drive and
downloads it, all in one cell, so a dropped session after it finishes cannot lose anything. Tier 0 logs every step,
sizes the unit of work to this GPU, locks the trial's clocks at a clock the probe saw held, restores the GPU's clocks
and power limit on exit, and writes `SUMMARY.md` and `tier0_bundle.tar.gz`. The bundle leaves out the run's private
key (`signing_key.bin`) on purpose: never share that file. If the browser asks whether to allow several downloads,
allow them.

In [ ]:
# 3. The run: Tier 0, the mock training run, then save and download.
import glob, hashlib
from IPython.display import Markdown, display
PROFILE = "medium"    # "medium": about 5 h, can decide the registered rules. "smoke": about 2 h 20 min, a pipeline check.
OUT = time.strftime("/content/plan-a-closure/runs/tier0-%Y%m%d-%H%M%S")   # a fresh folder for every run
!closure tier0 --confirm-hardware --gpu 0 --trial-profile {PROFILE} --out {OUT}
FIRST = None
if os.path.exists(f"{OUT}/logs/01-unit-tests.log") and not os.path.exists(f"{OUT}/logs/02-selftest.log"):
    # It stopped at its first step: the unit tests failed on this machine. That log is kept and the run starts again
    # without them, so that the session is not lost. Say so when you report the run.
    FIRST, OUT = OUT, OUT + "-b"
    print("The unit tests failed here; their log is saved as first-attempt-unit-tests.log. Running again without them.")
    !closure tier0 --confirm-hardware --gpu 0 --trial-profile {PROFILE} --skip-tests --out {OUT}
    shutil.copy2(f"{FIRST}/logs/01-unit-tests.log", f"{OUT}/first-attempt-unit-tests.log")
if os.path.exists(f"{OUT}/probe.json"):
    !closure treaty mock-run --device cuda:0 --probe {OUT}/probe.json --budget-flop 1e16 --dim 4096 --layers 8 --batch 2048 --out {OUT}/mock-run
keep = [p for p in (f"{OUT}/tier0_bundle.tar.gz", f"{OUT}/SUMMARY.md", f"{OUT}/mock-run/drill.json",
                    f"{OUT}/first-attempt-unit-tests.log") if os.path.exists(p)]
if SAVE:
    dest = os.path.join(SAVE, os.path.basename(OUT))
    os.makedirs(dest, exist_ok=True)
    for p in keep:
        shutil.copy2(p, dest)
    print("saved to Google Drive:", dest)
if os.path.exists(f"{OUT}/tier0_bundle.tar.gz"):
    print("bundle sha256:", hashlib.sha256(open(f"{OUT}/tier0_bundle.tar.gz", "rb").read()).hexdigest())
if os.path.exists(f"{OUT}/SUMMARY.md"):
    display(Markdown(open(f"{OUT}/SUMMARY.md", encoding="utf-8").read()))
from google.colab import files
for p in keep:
    files.download(p)

## After the run

* Send back `tier0_bundle.tar.gz`, `SUMMARY.md` and `drill.json` (in your Downloads folder, and in Google Drive under
  `plan-a-closure-runs/`). On Windows, `Get-FileHash $HOME\Downloads\tier0_bundle.tar.gz` should match the sha256 printed
  above.
* Publish the result whatever it shows, labelled as a shakedown: a new section in `docs/DEVIATIONS.md` (date, GPU,
  "Google Colab VM", no independent meter, the bundle's SHA-256) and the bundle under `results/`.
* `SUMMARY.md` reports which commit ran, the trial's SM clock and seed, the hidden load's own power, what the GPU did
  at each load level, and the pre-registered decision rules read at this run's scale (`closure criteria`). The medium
  trial can decide H1.1 to H1.4; the smoke trial cannot (40 null blocks, and the false-alarm rule needs 200).
* The pre-registration was stamped with OpenTimestamps before the trial began (`T1-medium/prereg.json.ots` in the
  bundle). A few hours later `ots upgrade prereg.json.ots` and `ots verify prereg.json.ots` tie it to a Bitcoin block.
* The next step is the registered run: Tier 0 and then Tier 1 on instrumented H100/H200 hardware
  (`docs/HARDWARE_CONFIG.md`).

Optional: a two-minute rehearsal on fake hardware. Set `REHEARSE = True` and run the cell; Run all skips it.

In [ ]:
REHEARSE = False
if REHEARSE:
    !closure tier0 --mock --skip-tests --out runs/rehearsal-{int(time.time())}